In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   SARCASM DETECTOR  —  FILE 2: HUGGINGFACE SPACES DASHBOARD               ║
# ║                                                                              ║
# ║   Deploy this as a HuggingFace Space:                                       ║
# ║     1. Go to huggingface.co/new-space                                       ║
# ║     2. SDK: Gradio  |  Hardware: CPU Basic (free) or T4 for faster LIME    ║
# ║     3. Upload this file as app.py                                            ║
# ║     4. Set MODEL_REPO below to your trained model repo                      ║
# ║                                                                              ║
# ║   OR run in Colab (File 2 mode):                                            ║
# ║     1. Paste entire file into one cell → Run                                ║
# ║     2. It auto-loads model from HF Hub or Google Drive                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Set this to your trained model repo (output from File 1 training)
MODEL_REPO  = "AK-Rahul/sarcasm-roberta"           # ← your trained model

# If running in Colab and model is on Drive instead of Hub:
DRIVE_MODEL = "/content/drive/MyDrive/totally-not-sarcastic"
USE_DRIVE   = False   # ← set True to load from Drive instead of HF Hub

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  INSTALL
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys
for pkg in ["transformers>=4.40", "gradio>=4.0", "plotly", "lime", "torch"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

# ═══════════════════════════════════════════════════════════════════════════════
# 2.  IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import os, warnings, io, csv, json
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3.  LOAD MODEL
# ═══════════════════════════════════════════════════════════════════════════════
print("\n🧠  Loading model…")
MODEL_PATH = DRIVE_MODEL if USE_DRIVE else MODEL_REPO

# If on Colab and using Drive, mount it first
if USE_DRIVE and not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, ignore_mismatched_sizes=True)
model.to(device)
model.eval()

MAX_LEN  = 128
N_PARAMS = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅  Model loaded from: {MODEL_PATH}  ({N_PARAMS:.0f}M params)")

# ── Load training metadata if available ──────────────────────────────────────
META_PATH = os.path.join(MODEL_PATH if USE_DRIVE else ".", "training_meta.json")
META = {}
if os.path.exists(META_PATH):
    with open(META_PATH) as f:
        META = json.load(f)
    print(f"✅  Metadata loaded  (val F1={META.get('f1', '?')})")
else:
    # Try downloading from HF Hub
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=MODEL_REPO, filename="training_meta.json")
        with open(p) as f:
            META = json.load(f)
        print(f"✅  Metadata from Hub  (val F1={META.get('f1', '?')})")
    except Exception:
        print("⚠️  training_meta.json not found — stats tab will show placeholders")

# ═══════════════════════════════════════════════════════════════════════════════
# 4.  INFERENCE HELPERS
# ═══════════════════════════════════════════════════════════════════════════════
LABELS     = ["Not Sarcastic 🙂", "Sarcastic 😏"]
LABEL_COLS = ["#34d399", "#f87171"]
_history   = []

def _proba(texts):
    """LIME-compatible: list[str] → (N, 2) probabilities."""
    enc = tokenizer(texts, truncation=True, padding=True,
                    max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=1).cpu().numpy()

def _infer(context: str, response: str):
    ctx = context.strip()
    if ctx:
        enc = tokenizer(ctx, response.strip(), truncation="longest_first",
                        padding=True, max_length=MAX_LEN, return_tensors="pt")
    else:
        enc = tokenizer(response.strip(), truncation=True,
                        padding=True, max_length=MAX_LEN, return_tensors="pt")
    enc = enc.to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs    = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    return pred_idx, float(probs[pred_idx]), probs

def _intensity(p: float):
    if p >= 0.92: return "🔥 Scorching",      "#ef4444"
    if p >= 0.78: return "😬 High Sarcasm",    "#f97316"
    if p >= 0.58: return "🤔 Moderate",        "#eab308"
    if p >= 0.42: return "😐 Borderline",      "#94a3b8"
    if p >= 0.22: return "🙂 Probably Sincere","#4ade80"
    return               "✅ Clearly Sincere",  "#34d399"

# ═══════════════════════════════════════════════════════════════════════════════
# 5.  PREDICTION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════
def predict_sarcasm(context: str, response: str):
    if not response.strip():
        warn = ("<div style='font-family:\"DM Mono\",monospace;color:#fbbf24;"
                "padding:1.2rem;background:#1c1917;border-radius:14px;"
                "border:1px solid #fbbf24;text-align:center'>"
                "⚠️  Enter a message to classify.</div>")
        return warn, None, pd.DataFrame(_history)

    pred_idx, conf, probs = _infer(context, response)
    label     = LABELS[pred_idx]
    color     = LABEL_COLS[pred_idx]
    i_label, i_color = _intensity(float(probs[1]))

    # ── Dial SVG ──────────────────────────────────────────────────────────────
    # A semicircular gauge showing sarcasm probability
    p_sarc = float(probs[1])
    angle  = -180 + p_sarc * 180   # maps 0→-180deg, 1→0deg
    import math
    rad    = math.radians(angle)
    nx     = 100 + 80 * math.cos(rad)
    ny     = 100 - 80 * math.sin(rad)  # SVG y-axis is inverted

    dial_svg = f"""<svg viewBox="0 0 200 110" xmlns="http://www.w3.org/2000/svg"
      style="width:100%;max-width:220px;display:block;margin:0 auto">
  <defs>
    <linearGradient id="g" x1="0%" y1="0%" x2="100%" y2="0%">
      <stop offset="0%"   stop-color="#34d399"/>
      <stop offset="50%"  stop-color="#eab308"/>
      <stop offset="100%" stop-color="#ef4444"/>
    </linearGradient>
  </defs>
  <!-- Track -->
  <path d="M 20 100 A 80 80 0 0 1 180 100" fill="none"
        stroke="#1e293b" stroke-width="14" stroke-linecap="round"/>
  <!-- Value arc -->
  <path d="M 20 100 A 80 80 0 0 1 180 100" fill="none"
        stroke="url(#g)" stroke-width="14" stroke-linecap="round"
        stroke-dasharray="251.3" stroke-dashoffset="{251.3*(1-p_sarc):.1f}"/>
  <!-- Needle -->
  <line x1="100" y1="100" x2="{nx:.1f}" y2="{ny:.1f}"
        stroke="{color}" stroke-width="3" stroke-linecap="round"/>
  <circle cx="100" cy="100" r="5" fill="{color}"/>
  <!-- Centre label -->
  <text x="100" y="88" text-anchor="middle"
        font-family="DM Mono,monospace" font-size="18" font-weight="900"
        fill="{color}">{p_sarc*100:.0f}%</text>
  <text x="100" y="108" text-anchor="middle"
        font-family="DM Mono,monospace" font-size="7" fill="#6b7280">SARCASM PROBABILITY</text>
</svg>"""

    # ── Result card ───────────────────────────────────────────────────────────
    ctx_echo = context.strip() if context.strip() else "<i style='color:#4b5563'>none</i>"
    result_html = (
        '<div style="font-family:\'DM Mono\',monospace;padding:1.6rem;border-radius:20px;'
        'background:linear-gradient(160deg,#0d1117,#161b22);'
        'border:2px solid ' + color + ';position:relative;overflow:hidden">'

        # glow accent
        '<div style="position:absolute;top:-40px;right:-40px;width:180px;height:180px;'
        'border-radius:50%;background:' + color + '18;pointer-events:none"></div>'

        # verdict
        '<div style="font-size:2.2rem;font-weight:900;color:' + color + ';'
        'letter-spacing:-.03em;line-height:1;text-shadow:0 0 30px ' + color + '44">'
        + label + '</div>'

        # intensity pill
        '<div style="display:inline-flex;align-items:center;gap:.4rem;margin-top:.6rem;'
        'padding:.25rem .8rem;background:' + i_color + '18;'
        'border:1px solid ' + i_color + '55;border-radius:999px;'
        'color:' + i_color + ';font-size:.8rem;font-weight:600">'
        + i_label + '</div>'

        # dial
        '<div style="margin:1rem 0">' + dial_svg + '</div>'

        # bars
        '<div style="font-size:.68rem;color:#4b5563;letter-spacing:.1em;'
        'text-transform:uppercase;margin-bottom:.5rem">Breakdown</div>'

        '<div style="display:flex;align-items:center;gap:.7rem;margin-bottom:.4rem">'
        '<div style="width:76px;color:#9ca3af;font-size:.78rem">😏 Sarcastic</div>'
        '<div style="flex:1;background:#0f172a;border-radius:4px;height:8px;overflow:hidden">'
        '<div style="width:' + f"{probs[1]*100:.1f}" + '%;background:#f87171;height:100%;'
        'border-radius:4px"></div></div>'
        '<div style="width:40px;text-align:right;color:#f87171;font-size:.85rem;font-weight:700">'
        + f"{probs[1]*100:.1f}" + '%</div></div>'

        '<div style="display:flex;align-items:center;gap:.7rem">'
        '<div style="width:76px;color:#9ca3af;font-size:.78rem">🙂 Sincere</div>'
        '<div style="flex:1;background:#0f172a;border-radius:4px;height:8px;overflow:hidden">'
        '<div style="width:' + f"{probs[0]*100:.1f}" + '%;background:#34d399;height:100%;'
        'border-radius:4px"></div></div>'
        '<div style="width:40px;text-align:right;color:#34d399;font-size:.85rem;font-weight:700">'
        + f"{probs[0]*100:.1f}" + '%</div></div>'

        # input echo
        '<div style="margin-top:1.1rem;padding:.8rem;background:#0d1117;border-radius:10px;'
        'font-size:.78rem;color:#4b5563;line-height:1.8">'
        '<span style="color:#818cf8;font-weight:600">CTX</span> ' + ctx_echo + '<br>'
        '<span style="color:#818cf8;font-weight:600">MSG</span> '
        + response.strip() + '</div>'
        '</div>'
    )

    # ── LIME ──────────────────────────────────────────────────────────────────
    lime_fig = None
    try:
        sep = getattr(tokenizer, "sep_token", "</s>")
        lime_text = (f"{context.strip()} {sep} {response.strip()}"
                     if context.strip() else response.strip())
        exp = LimeTextExplainer(class_names=["Sincere","Sarcastic"]).explain_instance(
            lime_text, _proba, num_features=14, num_samples=400, labels=[1])
        words_w = exp.as_list(label=1)
        words, weights = zip(*words_w)
        max_abs = max(abs(w) for w in weights) or 1
        bar_colors = [
            f"rgba(248,113,113,{min(1,abs(w)/max_abs*.85+.15):.2f})" if w > 0
            else f"rgba(52,211,153,{min(1,abs(w)/max_abs*.85+.15):.2f})"
            for w in weights
        ]
        lime_fig = go.Figure(go.Bar(
            x=list(weights), y=list(words), orientation="h",
            marker=dict(color=bar_colors, line=dict(width=0)),
            text=[f"{w:+.3f}" for w in weights], textposition="outside",
            textfont=dict(family="DM Mono", size=11, color="#94a3b8"),
        ))
        lime_fig.update_layout(
            title=dict(
                text="<b>Word Attribution</b>  "
                     "<span style='font-size:11px;color:#4b5563'>"
                     "🔴 → Sarcastic · 🟢 → Sincere</span>",
                font=dict(family="DM Mono", size=13, color="#e2e8f0")),
            height=420,
            plot_bgcolor="#0d1117", paper_bgcolor="#0d1117",
            font=dict(family="DM Mono", size=11, color="#94a3b8"),
            xaxis=dict(zeroline=True, zerolinecolor="#1e293b", zerolinewidth=2,
                       gridcolor="#0f172a", tickfont=dict(color="#4b5563")),
            yaxis=dict(autorange="reversed", tickfont=dict(color="#e2e8f0")),
            margin=dict(l=10, r=50, t=50, b=10),
        )
    except Exception:
        pass

    _history.append({
        "Context":    (context[:50]+"…") if len(context)>50 else (context or "—"),
        "Message":    (response[:60]+"…") if len(response)>60 else response,
        "Prediction": label,
        "Confidence": f"{conf*100:.1f}%",
        "Sarc %":     f"{probs[1]*100:.1f}%",
    })
    return result_html, lime_fig, pd.DataFrame(_history)


def batch_predict(context: str, messages: str):
    rows = [m.strip() for m in messages.splitlines() if m.strip()]
    if not rows: return pd.DataFrame()
    ctx = context.strip()
    if ctx:
        enc = tokenizer([ctx]*len(rows), rows, truncation="longest_first",
                        padding=True, max_length=MAX_LEN, return_tensors="pt")
    else:
        enc = tokenizer(rows, truncation=True, padding=True,
                        max_length=MAX_LEN, return_tensors="pt")
    enc = enc.to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs_all = torch.softmax(logits, dim=1).cpu().numpy()
    records = []
    for msg, probs in zip(rows, probs_all):
        idx = int(np.argmax(probs))
        il, _ = _intensity(float(probs[1]))
        records.append({"Message": msg, "Verdict": LABELS[idx], "Intensity": il,
                        "Sarc %": f"{probs[1]*100:.1f}%",
                        "Sincere %": f"{probs[0]*100:.1f}%",
                        "Confidence": f"{probs[idx]*100:.1f}%"})
    return pd.DataFrame(records)


def export_history():
    if not _history: return None
    path = "/tmp/sarcasm_predictions.csv"
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(_history[0].keys()))
        w.writeheader(); w.writerows(_history)
    return path


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  UI
# ═══════════════════════════════════════════════════════════════════════════════

# ── Pull real values from metadata ───────────────────────────────────────────
_acc     = META.get("accuracy", 0) * 100
_f1      = META.get("f1",       0) * 100
_prec_v  = META.get("precision",0) * 100
_rec     = META.get("recall",   0) * 100
_mdl     = META.get("model_name",  CFG["model_name"] if "CFG" in dir() else "roberta-base")
_epochs  = META.get("epochs",   3)
_lr      = META.get("lr",       "2e-5")
_batch   = META.get("batch",    32)
_ls      = META.get("label_smooth", 0.05)
_total   = META.get("total_samples", "—")
_sources = META.get("sources",  ["—"])
_nparams = META.get("n_params_M", round(N_PARAMS, 1))
_prec_s  = META.get("precision_str", "fp16")
_ctx     = META.get("has_context", 0)
_cm_data = META.get("confusion_matrix", [[0,0],[0,0]])

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:ital,wght@0,300;0,400;0,500;1,400&family=Syne:wght@700;800;900&display=swap');
*, *::before, *::after { box-sizing:border-box; }
html, body { background:#080c14 !important; }
.gradio-container {
  background:#080c14 !important;
  max-width:1500px !important;
  font-family:'DM Mono',monospace !important;
}
#MainMenu, footer, header { visibility:hidden; }

/* ── Tabs ── */
.tab-nav { background:transparent !important; border-bottom:1px solid #1e2d3d !important; }
.tab-nav button {
  font-family:'DM Mono',monospace !important;
  color:#4b5563 !important; font-size:.82rem !important;
  letter-spacing:.04em !important;
}
.tab-nav button.selected {
  color:#818cf8 !important;
  border-bottom:2px solid #6366f1 !important;
}

/* ── Inputs ── */
textarea, input[type=text] {
  font-family:'DM Mono',monospace !important;
  background:#0d1117 !important;
  color:#e2e8f0 !important;
  border:1px solid #1e2d3d !important;
  border-radius:10px !important;
  caret-color:#818cf8;
}
textarea:focus, input[type=text]:focus {
  border-color:#6366f1 !important;
  box-shadow:0 0 0 3px rgba(99,102,241,0.15) !important;
  outline:none !important;
}

/* ── Buttons ── */
.gr-button, button.lg, button.primary {
  font-family:'DM Mono',monospace !important;
  font-weight:500 !important;
  background:linear-gradient(135deg,#4f46e5,#6366f1) !important;
  color:#fff !important;
  border:none !important;
  border-radius:10px !important;
  letter-spacing:.04em !important;
  transition:all .2s ease !important;
}
.gr-button:hover { transform:translateY(-2px) !important;
  box-shadow:0 8px 24px rgba(99,102,241,0.4) !important; }

/* ── Labels ── */
label, .gr-form label {
  color:#4b5563 !important;
  font-size:.72rem !important;
  letter-spacing:.1em !important;
  text-transform:uppercase !important;
  font-weight:500 !important;
}

/* ── Cards / blocks ── */
.block {
  background:#0d1117 !important;
  border:1px solid #1e2d3d !important;
  border-radius:16px !important;
}

/* ── Tables ── */
table { background:#0d1117 !important; color:#cbd5e1 !important;
        font-size:.78rem !important; font-family:'DM Mono',monospace !important; }
th { background:#161b22 !important; color:#818cf8 !important; font-weight:600 !important; }
tr:nth-child(even) { background:#0f1923 !important; }
"""

# ── Header ────────────────────────────────────────────────────────────────────
_f1_str  = f"{_f1:.1f}%" if _f1 else "—"
_acc_str = f"{_acc:.1f}%" if _acc else "—"

HEADER = (
    '<div style="padding:1.8rem 0 1.2rem;border-bottom:1px solid #1e2d3d;margin-bottom:1.2rem">'
    '<div style="font-family:\'Syne\',sans-serif;font-size:3rem;font-weight:900;'
    'color:#f0f6ff;letter-spacing:-.05em;line-height:1">'
    '😏 totally<span style="color:#6366f1">-</span>not-sarcastic</div>'
    '<div style="margin-top:.6rem;font-family:\'DM Mono\',monospace;'
    'font-size:.82rem;color:#374151;display:flex;flex-wrap:wrap;gap:1.2rem">'
    '<span>' + _mdl + '</span>'
    '<span style="color:#1e2d3d">|</span>'
    '<span>Reddit SARC · TweetEval · Headlines</span>'
    '<span style="color:#1e2d3d">|</span>'
    '<span style="color:#34d399">accuracy ' + _acc_str + '</span>'
    '<span style="color:#1e2d3d">|</span>'
    '<span style="color:#f87171">F1 ' + _f1_str + '</span>'
    '<span style="color:#1e2d3d">|</span>'
    '<span>LIME explainability</span>'
    '</div></div>'
)

EXAMPLES = [
    ["How was the surgery?",       "Oh perfect, I only lost feeling in my left hand."],
    ["Did you enjoy the concert?", "Every off-key note was a gift to my ears."],
    ["How was traffic today?",     "Two hours for 10 km. Really efficient city planning."],
    ["",                           "I absolutely love Mondays. Best day of the week."],
    ["Are you ready for the exam?","Never been more prepared in my entire life."],
    ["Did the presentation go well?","Yes, the client signed immediately — it was great."],
    ["",                           "Scientists say sleeping more is the best productivity hack."],
    ["How was the flight?",        "Oh wonderful, only delayed by four hours."],
]

with gr.Blocks(theme=gr.themes.Base(), css=CSS, title="😏 totally-not-sarcastic") as demo:

    gr.HTML(HEADER)

    with gr.Tabs():

        # ── Tab 1: Predict ────────────────────────────────────────────────────
        with gr.Tab("/ predict"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=360):
                    ctx_box = gr.Textbox(
                        label="Context  (previous message — optional)",
                        placeholder="e.g.  How was your day?",
                        lines=3, max_lines=6)
                    resp_box = gr.Textbox(
                        label="Message  ← classify this",
                        placeholder="e.g.  Just the best day of my entire life.",
                        lines=3, max_lines=6)
                    pred_btn = gr.Button("→  Detect Sarcasm", variant="primary", size="lg")
                    gr.HTML(
                        '<div style="color:#374151;font-size:.72rem;margin-top:.4rem;'
                        'font-family:\'DM Mono\',monospace">'
                        '💡 Adding context significantly improves accuracy for conversational sarcasm'
                        '</div>')
                    gr.Examples(examples=EXAMPLES, inputs=[ctx_box, resp_box],
                                label="Quick examples", examples_per_page=8)

                with gr.Column(scale=1, min_width=360):
                    result_html = gr.HTML(
                        '<div style="font-family:\'DM Mono\',monospace;color:#374151;'
                        'font-size:.85rem;padding:2.5rem;text-align:center;'
                        'border:1px dashed #1e2d3d;border-radius:18px;'
                        'background:#0d1117">'
                        '↑ enter a message and hit detect</div>')
                    lime_plot = gr.Plot()

            # History
            gr.HTML('<div style="height:1px;background:#1e2d3d;margin:1rem 0"></div>')
            with gr.Row():
                gr.HTML(
                    '<div style="font-family:\'DM Mono\',monospace;color:#4b5563;'
                    'font-size:.72rem;letter-spacing:.1em;text-transform:uppercase;'
                    'padding:.4rem 0">📋 Prediction History</div>')
                export_btn = gr.Button("↓ Export CSV", size="sm")
            hist_table  = gr.Dataframe(label="", wrap=True,
                headers=["Context","Message","Prediction","Confidence","Sarc %"])
            export_file = gr.File(label="Download", visible=False)

            pred_btn.click(fn=predict_sarcasm,
                           inputs=[ctx_box, resp_box],
                           outputs=[result_html, lime_plot, hist_table])
            export_btn.click(fn=export_history, outputs=[export_file])
            export_file.change(fn=lambda f: gr.update(visible=f is not None),
                               inputs=[export_file], outputs=[export_file])

        # ── Tab 2: Batch ──────────────────────────────────────────────────────
        with gr.Tab("/ batch"):
            gr.HTML(
                '<div style="font-family:\'DM Mono\',monospace;color:#4b5563;'
                'font-size:.82rem;padding:.8rem 0 1.2rem">'
                'Classify multiple messages at once. One message per line. '
                'Optional shared context applied to all.</div>')
            batch_ctx  = gr.Textbox(label="Shared context (optional)", lines=2,
                                    placeholder="e.g.  How was the holiday?")
            batch_msgs = gr.Textbox(
                label="Messages — one per line", lines=10,
                placeholder=(
                    "Oh just absolutely wonderful\n"
                    "The beach was nice, I enjoyed it\n"
                    "Sure, I love losing my luggage\n"
                    "Flight was actually on time for once — miracle"))
            batch_btn = gr.Button("→  Classify All", variant="primary")
            batch_out = gr.Dataframe(label="Results", wrap=True)
            batch_btn.click(fn=batch_predict,
                            inputs=[batch_ctx, batch_msgs], outputs=batch_out)

        # ── Tab 3: Model Stats ────────────────────────────────────────────────
        with gr.Tab("/ model_stats"):

            # Confusion matrix
            cm_fig = go.Figure(go.Heatmap(
                z=_cm_data,
                x=["Predicted: Sincere", "Predicted: Sarcastic"],
                y=["Actual: Sincere",    "Actual: Sarcastic"],
                colorscale=[[0,"#080c14"],[0.4,"#312e81"],[1,"#f87171"]],
                text=[[str(v) for v in row] for row in _cm_data],
                texttemplate="<b>%{text}</b>", showscale=True,
            ))
            cm_fig.update_layout(
                title=dict(text="<b>Confusion Matrix</b>",
                           font=dict(family="DM Mono", color="#e2e8f0")),
                height=400, plot_bgcolor="#0d1117", paper_bgcolor="#0d1117",
                font=dict(family="DM Mono", color="#94a3b8"),
                xaxis=dict(tickfont=dict(color="#94a3b8")),
                yaxis=dict(tickfont=dict(color="#94a3b8")),
                margin=dict(l=10,r=10,t=50,b=10),
            )

            # Radar chart
            radar_vals = [_acc, _f1, _prec_v, _rec]
            radar_fig  = go.Figure(go.Scatterpolar(
                r=radar_vals + [radar_vals[0]],
                theta=["Accuracy","F1","Precision","Recall","Accuracy"],
                fill="toself",
                fillcolor="rgba(99,102,241,0.18)",
                line=dict(color="#6366f1", width=2),
                marker=dict(color="#818cf8", size=7),
            ))
            radar_fig.update_layout(
                polar=dict(
                    bgcolor="#0d1117",
                    radialaxis=dict(range=[0,100], tickfont=dict(color="#374151",size=8),
                                    gridcolor="#1e2d3d"),
                    angularaxis=dict(tickfont=dict(color="#e2e8f0",size=11),
                                     gridcolor="#1e2d3d"),
                ),
                paper_bgcolor="#0d1117",
                font=dict(family="DM Mono", color="#94a3b8"),
                height=360, margin=dict(l=50,r=50,t=20,b=20),
            )

            # ── Metric cards ─────────────────────────────────────────────────
            _mc = ""
            for _nm, _vl, _c, _ico in [
                ("Accuracy",  _acc,   "#818cf8", "🎯"),
                ("F1 Score",  _f1,    "#f87171", "⚡"),
                ("Precision", _prec_v,"#34d399", "🔍"),
                ("Recall",    _rec,   "#38bdf8", "📡"),
            ]:
                _vl_str = f"{_vl:.1f}" if _vl else "—"
                _mc += (
                    '<div style="background:#0d1117;border:1px solid #1e2d3d;'
                    'border-top:3px solid ' + _c + ';border-radius:14px;'
                    'padding:1.3rem;text-align:center">'
                    '<div style="font-size:1.6rem">' + _ico + '</div>'
                    '<div style="font-size:1.9rem;font-weight:700;color:' + _c + ';'
                    'letter-spacing:-.03em;margin:.2rem 0">' + _vl_str +
                    '<span style="font-size:.9rem">%</span></div>'
                    '<div style="color:#374151;font-size:.65rem;letter-spacing:.12em;'
                    'text-transform:uppercase">' + _nm + '</div></div>'
                )

            # ── Architecture table ────────────────────────────────────────────
            _ar = ""
            for _k, _v in [
                ("Base Model",       f"{_mdl}  ({_nparams}M params)"),
                ("Training",         f"{_epochs} epochs · LR={_lr} · cosine decay · batch={_batch}"),
                ("Regularisation",   f"label_smooth={_ls} · weight_decay=0.01 · grad_clip=1.0"),
                ("Precision",        f"{_prec_s}  (fp16 disabled for DeBERTa-v3)"),
                ("Early Stopping",   "patience=2 epochs on eval F1"),
                ("Datasets",         " + ".join(_sources) if _sources != ["—"] else "Reddit SARC + TweetEval + Headlines"),
                ("Total Samples",    f"{_total:,}" if isinstance(_total, int) else str(_total)),
                ("Context Pairs",    f"{_ctx:,}" if isinstance(_ctx, int) else str(_ctx)),
                ("Context Strategy", "text_a=parent_comment · truncation=longest_first"),
                ("Explainability",   "LIME  (14 features · 400 perturbation samples)"),
            ]:
                _ar += (
                    '<tr><td style="padding:.4rem .8rem;color:#374151;width:32%;'
                    'font-size:.75rem;border-bottom:1px solid #0d1117">' + _k + '</td>'
                    '<td style="padding:.4rem .8rem;color:#cbd5e1;font-size:.75rem;'
                    'border-bottom:1px solid #0d1117">' + str(_v) + '</td></tr>'
                )

            stats_html = (
                '<div style="display:grid;grid-template-columns:repeat(4,1fr);'
                'gap:.8rem;margin:1rem 0;font-family:\'DM Mono\',monospace">'
                + _mc + '</div>'
                '<div style="font-family:\'DM Mono\',monospace;background:#0d1117;'
                'border:1px solid #1e2d3d;border-radius:14px;padding:1.3rem;margin-top:.8rem">'
                '<div style="color:#6366f1;font-size:.65rem;letter-spacing:.14em;'
                'font-weight:600;text-transform:uppercase;margin-bottom:.8rem">'
                '⚙ Architecture</div>'
                '<table style="width:100%;border-collapse:collapse">' + _ar + '</table>'
                '</div>'
            )

            gr.HTML(stats_html)
            with gr.Row():
                gr.Plot(value=cm_fig)
                gr.Plot(value=radar_fig)

    # ── Footer ────────────────────────────────────────────────────────────────
    gr.HTML(
        '<div style="margin-top:1.5rem;padding:1rem;'
        'border-top:1px solid #1e2d3d;text-align:center;'
        'font-family:\'DM Mono\',monospace;font-size:.72rem;color:#374151">'
        'RoBERTa-base · Reddit SARC (Khodak et al.) · TweetEval · '
        'News Headlines · LIME · Gradio · Plotly'
        '</div>'
    )

# ═══════════════════════════════════════════════════════════════════════════════
# 7.  LAUNCH
# ═══════════════════════════════════════════════════════════════════════════════
# Detect environment
_in_colab = "google.colab" in str(globals().get("__builtins__",""))
try:
    from google.colab import output as _co
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    print("\n🚀  Launching totally-not-sarcastic (Colab)…")
    demo.launch(share=True, debug=False)
else:
    # HuggingFace Spaces / local
    print("\n🚀  Launching totally-not-sarcastic (Spaces/local)…")
    demo.launch()